# Tool useage

In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


In [ ]:
import os
import json
from urllib.parse import urlencode
from urllib.request import urlopen
from openai import OpenAI


client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)


def city_to_coordinates(city: str) -> dict:
    url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode({
        "name": city,
        "count": 1,
        "language": "zh",
        "format": "json",
    })

    with urlopen(url, timeout=10) as response:
        data = json.loads(response.read().decode("utf-8"))

    if not data.get("results"):
        return {"error": "city not found"}

    location = data["results"][0]

    return {
        "city": location["name"],
        "country": location.get("country"),
        "latitude": location["latitude"],
        "longitude": location["longitude"],
    }


def get_weather_by_coordinates(latitude: float, longitude: float) -> dict:
    url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "auto",
    })

    with urlopen(url, timeout=10) as response:
        data = json.loads(response.read().decode("utf-8"))

    current = data["current"]

    return {
        "time": current["time"],
        "temperature": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "precipitation": current["precipitation"],
        "wind_speed": current["wind_speed_10m"],
        "unit": {
            "temperature": "°C",
            "humidity": "%",
            "precipitation": "mm",
            "wind_speed": "km/h",
        },
    }


tools = [
    {
        "type": "function",
        "function": {
            "name": "city_to_coordinates",
            "description": "根据城市名称查询城市的经纬度",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 深圳、北京、New York",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "根据经纬度查询当前天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "纬度",
                    },
                    "longitude": {
                        "type": "number",
                        "description": "经度",
                    },
                },
                "required": ["latitude", "longitude"],
            },
        },
    },
]


available_tools = {
    "city_to_coordinates": city_to_coordinates,
    "get_weather_by_coordinates": get_weather_by_coordinates,
}


messages = [
    {
        "role": "system",
        "content": "你是一个天气助手。如果用户给的是城市名称，先调用 city_to_coordinates，再调用 get_weather_by_coordinates。",
    },
    {
        "role": "user",
        "content": "深圳现在天气怎么样？",
    },
]


for _ in range(5):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    assistant_message = response.choices[0].message
    messages.append(assistant_message.model_dump(exclude_none=True))

    if not assistant_message.tool_calls:
        print(assistant_message.content)
        break

    for tool_call in assistant_message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        result = available_tools[tool_name](**tool_args)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result, ensure_ascii=False),
        })